# 07_eval / qualitative_analysis.ipynb

**Purpose:** Pull out **interpretable example pairs** for the paper:
- True positives (correctly flagged OOC) with high gate weight on the explanatory stream.
- False positives (genuine pairs misclassified as OOC).
- False negatives (OOC missed by the model).
- True negatives.

For each, save a small montage with the image, caption, true label, predicted prob, and the gate weights (which stream voted strongest).

**Inputs:**
- `cache/fusion/fusion_predictions.csv` (must contain per-row gate weights `gate_a`, `gate_b`, `gate_c` if available)
- `data/processed/ooc_pairs/manifests/all.csv` — for caption text + image path

**Outputs:**
- `paper/tables/qualitative_examples.csv` — top-K examples per category with metadata
- `paper/figures/qualitative_TP.png`, `qualitative_FP.png`, `qualitative_FN.png`, `qualitative_TN.png` — image grids

**Notes:**
- Only NewsCLIPpings has limited local images, but VERITE & MMFakeBench do — examples are drawn preferentially from datasets with available local images.
- If gate columns are absent we fall back to ranking by predicted probability.


In [ ]:
# Imports
from __future__ import annotations
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from PIL import Image

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, PROCESSED_IMAGES, VERITE_IMAGES,
    PAPER, FIGURES, TABLES,
)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print("Repo root:", ROOT)


## 1. Load predictions + manifest, merge

In [ ]:
fusion_path = FUSION_CACHE / "fusion_predictions.csv"
thresh_path = FUSION_CACHE / "fusion_thresholds.json"
manifest = pd.read_csv(ALL_MANIFEST)
preds = pd.read_csv(fusion_path)

print("manifest cols:", list(manifest.columns)[:25])
print("preds cols   :", list(preds.columns)[:25])
print("manifest shape:", manifest.shape, "| preds shape:", preds.shape)

if thresh_path.exists():
    thresholds = json.loads(thresh_path.read_text())
else:
    thresholds = {}

def find_col(d, cands):
    for c in cands:
        if c in d.columns: return c
    return None

# pair_id key
pid = find_col(preds, ["pair_id", "id"])
y_col = find_col(preds, ["y_true", "label", "target"])
prob_col = find_col(preds, ["fusion_prob", "prob_fusion", "y_prob_fusion", "y_prob"])
ds_col_p = find_col(preds, ["dataset", "source_dataset"])
sp_col_p = find_col(preds, ["split", "data_split"])
ga_col = find_col(preds, ["gate_a", "ga", "g_a"])
gb_col = find_col(preds, ["gate_b", "gb", "g_b"])
gc_col = find_col(preds, ["gate_c", "gc", "g_c"])
print(f"y={y_col}, prob={prob_col}, gates=({ga_col},{gb_col},{gc_col})")

merge_cols = ["pair_id"]
mani_keep = [c for c in ["pair_id","caption","text","image_path","image","dataset","split","label"] if c in manifest.columns]
merged = preds.merge(manifest[mani_keep].drop_duplicates("pair_id"), on="pair_id", how="left", suffixes=("","_mf"))
print("merged shape:", merged.shape)
merged.head()


## 2. Pick threshold per dataset and tag confusion-matrix bucket

In [ ]:
# Determine dataset/split for each row
ds_eff  = merged[ds_col_p] if ds_col_p else merged.get("dataset")
sp_eff  = merged[sp_col_p] if sp_col_p else merged.get("split")
merged["_dataset"] = ds_eff
merged["_split"]   = sp_eff

def thresh_for_row(r):
    key = f"{r['_dataset']}_{r['_split']}"
    return float(thresholds.get(key, thresholds.get(f"{r['_dataset']}_test", thresholds.get("default", 0.5))))

merged["_threshold"] = merged.apply(thresh_for_row, axis=1)
merged["_y_pred"] = (merged[prob_col].astype(float) >= merged["_threshold"]).astype(int)
merged["_y_true"] = merged[y_col].astype(int)

def bucket(r):
    yt, yp = r["_y_true"], r["_y_pred"]
    if yt == 1 and yp == 1: return "TP"
    if yt == 0 and yp == 1: return "FP"
    if yt == 1 and yp == 0: return "FN"
    return "TN"

merged["_bucket"] = merged.apply(bucket, axis=1)
print(merged["_bucket"].value_counts())
print()
print("Per-dataset:")
print(merged.groupby(["_dataset","_split","_bucket"]).size().head(30))


## 3. Resolve image paths

In [ ]:
def resolve_image(row):
    """Return absolute Path to image if it exists locally, else None."""
    cand = row.get("image_path") or row.get("image")
    if pd.isna(cand) or not cand:
        return None
    p = Path(str(cand))
    if p.is_absolute() and p.exists():
        return p
    # try processed-images flat folder
    for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT / "data" / "raw" / "mmfakebench"]:
        cand2 = base / p.name
        if cand2.exists():
            return cand2
    # try as-given relative to ROOT
    cand3 = ROOT / p
    if cand3.exists():
        return cand3
    return None

merged["_image_path"] = merged.apply(resolve_image, axis=1)
print("Rows with local image:", merged["_image_path"].notna().sum())


## 4. Pick top-K examples per bucket (prefer with local image, prefer high-confidence)

In [ ]:
K = 6  # examples per bucket
TARGET_DATASETS = ["mmfakebench", "verite", "newsclippings"]  # prefer those with images

def score_example(row, bucket):
    """Higher score = more interesting / illustrative."""
    p = float(row[prob_col])
    if bucket == "TP": return p
    if bucket == "TN": return 1.0 - p
    if bucket == "FP": return p           # model is very confident but wrong
    if bucket == "FN": return 1.0 - p     # model very confident genuine but it was OOC
    return 0.0

selected = []
for bk in ["TP", "FP", "FN", "TN"]:
    cand = merged[(merged["_bucket"] == bk) & (merged["_split"] == "test")].copy()
    cand = cand[cand["_dataset"].isin(TARGET_DATASETS)]
    cand["_pref"] = cand["_image_path"].notna().astype(int)
    cand["_score"] = cand.apply(lambda r: score_example(r, bk), axis=1)
    cand = cand.sort_values(["_pref", "_score"], ascending=[False, False]).head(K)
    cand["_bucket"] = bk
    selected.append(cand)

selected_df = pd.concat(selected, ignore_index=True)
caption_col = "caption" if "caption" in selected_df.columns else ("text" if "text" in selected_df.columns else None)
keep_cols = ["_bucket", "pair_id", "_dataset", "_split", "_y_true", "_y_pred", prob_col, "_threshold"]
if caption_col: keep_cols.append(caption_col)
for g in [ga_col, gb_col, gc_col]:
    if g: keep_cols.append(g)
keep_cols.append("_image_path")

selected_df[keep_cols].to_csv(TABLES / "qualitative_examples.csv", index=False)
print("Saved:", TABLES / "qualitative_examples.csv")
selected_df[keep_cols].head(24)


## 5. Render image grids (one figure per bucket)

In [ ]:
def render_bucket(bucket_name):
    sub = selected_df[selected_df["_bucket"] == bucket_name].reset_index(drop=True)
    if sub.empty:
        print(f"No {bucket_name} rows.")
        return
    n = len(sub)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.4, rows * 4.4))
    axes = np.atleast_2d(axes)
    for k in range(rows * cols):
        ax = axes[k // cols, k % cols]
        ax.axis("off")
        if k >= n:
            continue
        r = sub.iloc[k]
        img_path = r["_image_path"]
        if pd.notna(img_path) and Path(img_path).exists():
            try:
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"[load err]\n{e}", ha="center", va="center", fontsize=8)
        else:
            ax.text(0.5, 0.5, "[no local image]", ha="center", va="center", fontsize=10, color="gray")
        # caption + meta
        cap = r.get(caption_col) if caption_col else ""
        cap = (str(cap)[:90] + "…") if isinstance(cap, str) and len(str(cap)) > 90 else cap
        meta = f"{r['_dataset']}  |  y={int(r['_y_true'])}, ŷ={int(r['_y_pred'])}, p={float(r[prob_col]):.3f}"
        if ga_col and gb_col and gc_col:
            meta += f"\ngates A={float(r[ga_col]):.2f}, B={float(r[gb_col]):.2f}, C={float(r[gc_col]):.2f}"
        ax.set_title(meta, fontsize=8)
        ax.text(0.0, -0.05, cap, transform=ax.transAxes, fontsize=8, va="top", wrap=True)
    fig.suptitle(f"Qualitative — {bucket_name}", fontsize=12, y=1.02)
    plt.tight_layout()
    out = FIGURES / f"qualitative_{bucket_name}.png"
    plt.savefig(out, bbox_inches="tight", dpi=140)
    print("Saved:", out)
    plt.show()

for bk in ["TP", "FP", "FN", "TN"]:
    render_bucket(bk)


## 6. Stream-attribution summary (which stream the gate trusts most)

In [ ]:
if all([ga_col, gb_col, gc_col]):
    test_only = merged[merged["_split"] == "test"].copy()
    gate_long = pd.melt(
        test_only,
        id_vars=["_dataset", "_bucket"],
        value_vars=[ga_col, gb_col, gc_col],
        var_name="stream",
        value_name="gate_weight",
    )
    gate_long["stream"] = gate_long["stream"].map({ga_col: "A", gb_col: "B", gc_col: "C"})
    summary = gate_long.groupby(["_dataset", "_bucket", "stream"])["gate_weight"].mean().unstack("stream")
    print(summary.round(3))
    summary.to_csv(TABLES / "qualitative_gate_attribution.csv", float_format="%.4f")
    print("Saved:", TABLES / "qualitative_gate_attribution.csv")
else:
    print("Gate columns not found in fusion_predictions.csv — skipping attribution summary.")


---
**Done.** Use these figures and the CSV in the paper's qualitative section. Cherry-pick the most pedagogical examples (clear OOC mismatches; illustrative failures).